# Manifestro Stage 1 Aether — LoquaciousSet Mimi cache
Запускайте ячейки строго сверху вниз. Код берётся из GitHub, результат сохраняется в Google Drive.

## 1. Проверка GPU

In [ ]:
!nvidia-smi

## 2. Клонирование кода

In [ ]:
%cd /content
!rm -rf /content/aether-dataset
!git clone https://github.com/karl4th/dataset-coll.git /content/aether-dataset
%cd /content/aether-dataset
!git rev-parse HEAD
!git status --short --branch

Для финального прогона желательно переключиться на проверенный commit: `!git checkout --detach <COMMIT_SHA>`. 

## 3. Установка окружения через uv

In [ ]:
%pip install -q uv
!uv sync --frozen --no-dev

## 4. Google Drive, HF token и output path

In [ ]:
import os
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Добавьте HF_TOKEN в Colab Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_CACHE'] = '/content/drive/MyDrive/.cache/huggingface/hub'
OUTPUT = Path('/content/drive/MyDrive/manifestro/stage1_aether_cache')
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG = Path('/content/aether-dataset/configs/loquacious-medium.yaml')
config_text = CONFIG.read_text()
config_text = config_text.replace('REQUIRED_OUTPUT_PATH', str(OUTPUT))
CONFIG.write_text(config_text)
print('Output:', OUTPUT)

## 5. Проверка окружения

In [ ]:
!uv run --frozen python -c "import sys, torch; print(sys.version); print('torch', torch.__version__); print('cuda', torch.version.cuda); print('available', torch.cuda.is_available()); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"
!uv run --frozen aether-dataset --help

## 6. Обязательный benchmark
Полный вывод виден в ячейке и сохраняется на Drive.

In [ ]:
!set -o pipefail; uv run --frozen aether-dataset benchmark --config configs/loquacious-medium.yaml 2>&1 | tee "$OUTPUT/benchmark.log"

При ошибке не запускайте полный прогон. Покажите последние строки следующей ячейки.

In [ ]:
!tail -n 100 "$OUTPUT/benchmark.log"

## 7. Полный restartable прогон

In [ ]:
!set -o pipefail; uv run --frozen aether-dataset run --config configs/loquacious-medium.yaml --resume 2>&1 | tee -a "$OUTPUT/run.log"

## 8. Статус и полная проверка

In [ ]:
!uv run --frozen aether-dataset status --output "$OUTPUT"
!uv run --frozen aether-dataset validate --output "$OUTPUT"

## 9. Публикация в приватный Hugging Face repo
Запускайте только после успешной полной валидации.

In [ ]:
!set -o pipefail; uv run --frozen aether-dataset publish --output "$OUTPUT" --repo manifestro/stage1_aether 2>&1 | tee -a "$OUTPUT/publish.log"